In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf
import pymc as pm
import arviz as az
import statsmodels.api as sm

### 04/13 

Notes on simulation, variance, bias:

Suppose we have two (unbiased) estimators for poisson:

$$ \hat \mu_1 = \bar x$$
$$ \hat \mu_2 = S^2$$

In [ ]:
# step 1: define parameters
mu = 5.3
n = 50
n_trials = 1000

# estimator 1: mean
estimator1 = []
# estimator 2: variance
estimator2 = []

for _ in range(n_trials):
    # do n poisson samples
    sample = np.random.poisson(mu, n)

    # update estimator 1
    estimator1.append(np.mean(sample))

    # update estimator 2
    estimator2.append(np.var(sample))

# histogram for both estimators, same bucket size, with dotted line for mu
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(estimator1, bins=10, density=True, color='blue', alpha=0.4, label='Mean')
ax.hist(estimator2, bins=10, density=True, color='red', alpha=0.4, label='Variance')
ax.axvline(mu, color='black', linestyle='--', label='True μ')

ax.set_title('Histograms of Mean and Variance', fontsize=13)
ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

If we have two unbiased estimators $\hat \theta_1$, $\hat \theta_2$, we prefer the one with the smaller variance.

But how can we know if $\hat \theta$ is the *best* unbiased estimator?

Result? Cramer-Rao Inequality

Lemma: $\mathbb E [\mathrm{score}] = 0$

Lemma: $\mathrm{Cov}[t(x), \ell'_X(\theta)] = 1 $

Theorem: Let $t(x)$ be an unbiased estimator. Then
$$
    \mathrm{Var}[t(x)] \geq \frac{1}{\mathrm{Var}[\ell'_X(\theta)]},
$$
giving us a lower bound on variance.

Proof: The second lemma gives us
$$
    1 = \mathrm{Cov}[t(x), \ell'_X(\theta)] 
$$
and after squaring both sides we have
$$
    1 = \mathrm{Cov}^2[t(x), \ell'_X(\theta)].
$$
Treating Cov as a dot product, the Cauchy-Schwarz inequality gives us
$$
1 = \mathrm{Cov}^2[t(x), \ell'_X(\theta)] \leq \mathrm{Var}[t(x)] \mathrm{Var}[\ell'_X(\theta)]
$$
Dividing by $\mathrm{Var}[\ell'_X(\theta)], we finally arrive at
$$
\frac{1}{\mathrm{Var}[\ell'_X(\theta)]} = \frac{\mathrm{Cov}^2[t(x), \ell'_X(\theta)]}{\mathrm{Var}[\ell'_X(\theta)]} \leq \mathrm{Var}[t(x)]
$$
or 
$$
\mathrm{Var}[t(x)] \geq \frac{1}{\mathrm{Var}[\ell'_X(\theta)]}
$$


### 04/15

Bias and variance for bayesian methods

Want to show

$$
\mathrm{Var}[\ell'_X (\theta)] = - \mathbb E [\ell_X '' (\theta)]
$$

1) Calculate MLE

$$
\ell(p) = \log {n \choose x} + x \log p + (n-x) \log (1-p)
$$
$$
\frac{\partial \ell}{\partial p} = \frac{x}{p} - \frac{n-x}{1-p} = 0
$$
$$
\frac{x}{p} = \frac{n-x}{1-p}
$$
$$
x(1-p) = p(n-x) = x - xp = np - xp
$$
$$
p = \frac {x}{n}
$$

2) Assuming prior = Beta(2, 2):

$$\frac{\alpha}{\alpha + \beta} = \frac{2 + x}{2 + x + 2 + (n - x)} = \frac{2 + x}{4 + n}$$


3) Are the estimators unbiased?

$$
\mathbb E[\frac{x}{n}] = \frac{1}{n} \mathbb E[X] = \frac{1}{n} pn = p 
$$

$$
\mathbb E[\frac{2 + x}{4 + n}] = \frac{1}{4 + n} \mathbb E[2 + x] = \frac{1}{4 + n} (2 + \mathbb E[x]) = \frac{2 + np }{4 + n}  \neq p
$$

4) If unbiased, is it the best estimator?
$$
    \ell'_x(p) = \frac{x}{p} - \frac{n-x}{1-p}
$$

\begin{align*}
    \mathrm{Var}[\ell'_x(p)] 
    &= \mathrm {Var}[\frac{x}{p} - \frac{n-x}{1-p}] \\
    &= \mathrm {Var}[\frac{x - np}{p(1-p)}] \\
    &= \frac{\mathrm {Var} [x - np]}{p^2(1-p)^2} \\
    &= \frac{\mathrm {Var} [x]}{p^2(1-p)^2} \\
    &- \frac{np(1-p)}{p^2(1-p)^2} \\
    &- \frac{n}{p(1-p)} \implies CRLB = \frac{p(1-p)} {n}
\end{align*}

On the other hand, we have
\begin{align*}
    \mathrm{Var}[\frac{x}{n}] 
    &= \frac{\mathrm{Var}[x]}{n^2} 
    &= \frac{np(1-p)}{n^2}  
    &= \frac{p(1-p)} {n}
\end{align*}

So MLE is the best unbiased estimator for $X \sim \mathrm{Bin}$.


## 04/22

Asymptotics Recap:

$$
X_1, \ldots, X_n \sim^{iid} f_\theta (x)
$$

We observe $n \rightarrow \infty$

Consistency of MLE:

$$
\hat \theta_{MLE} \rightarrow \theta_0 \text{ as } n \rightarrow \infty
$$

In Stats 118, there are two ways to prove consistency:
1. Law of Large Numbers
$$
\bar X \rightarrow \mu = \mathbb E[X_1]
$$

2. Continuous Mapping Theorem/Delta Method
$$
g(\bar X) \rightarrow g(\mu), \text{ where } \mu = \mathbb E[X_1]
$$

Example for exponential distribution: 
$$
\lambda = \frac{1}{\bar X} \rightarrow \frac{1}{E[X_1]} = \frac{1}{\mathbb E[X_1]} = \frac{1}{1 / \lambda_0} = \lambda_0
$$

But not sufficient for all MLEs! (ask about root finding algorithm)

Proof for consistency of MLE:

We define
\begin{align*}
\hat \theta_{MLE} 
&= \mathrm{argmax}_\theta \frac{1}{n} \ell_{X_1, \ldots, X_n} (\theta) \\
&= \mathrm{argmax}_\theta \frac{1}{n} \sum_{i = 1}^n \ell_{X_i} (\theta) \\
&\rightarrow \mathrm{argmax}_\theta \mathbb E_{\theta_0}[\ell_{X_i} (\theta)] & \text{by law of large numbers, under the true } \theta_0 \\
&= \theta_0
\end{align*}

And 

\begin{align*}

\mathbb E_{\theta_0} [\ell_{X_i} (\theta)] 
&= \int_{-\infty}^\infty \ell_X(\theta) f_{\theta_0} (x) \mathrm dx \\
&= \int_{-\infty}^\infty f_{\theta_0} (x) \log f_{\theta} (x) \mathrm dx &\text{(cross-entropy)} \\
&= \int_{-\infty}^\infty f_{\theta_0} (x) \log \frac{f_{\theta} (x)}{f_{\theta_0} (x)} \mathrm dx+ 
   \int_{-\infty}^\infty f_{\theta_0} (x) \log f_{\theta_0} (x) \mathrm dx \\
&= \text{-KL divergence between } f_{\theta_0} \text{ and } f_\theta + \text{ entropy of } f_{\theta_0}
\end{align*}

Now we want to prove that not only is MLE consistent, but it also conveges to a Normal dist.

Specifically,
$$
\sqrt{n} (\hat \theta_{MLE} - \theta_0) \implies \mathrm{Normal}(0, ?) \text{ as } n \rightarrow \infty
$$

We can use CLT. If $Y_1, \ldots, Y_n$ are i.i.d, then 
$$
\sqrt{n} (\bar Y - \mathbb E[Y_1]) \implies \mathrm{Normal}(0, \mathrm{Var}[Y_1])
$$

Theorem: $\hat \theta_{MLE}$ is asymtotically normal.

Proof:

Taylor expansion of $\ell'(\theta)$ around $\theta_0$:

$$
\ell'(\theta) \approx \ell'(\theta_0) + \ell''(\theta_0)(\hat \theta - \theta_0)
$$

If you plug in $\ell'(\theta_{MLE})$, then

$$
0 = \ell'(\theta) \approx \ell'(\theta_0) + \ell''(\theta_0)(\hat \theta - \theta_0)
$$

So 

$$
(\hat \theta - \theta_0) \approx \frac{\ell'(\theta_0)}{- \ell''(\theta_0)} 
= \frac{\frac 1n \ell'(\theta_0)}{- \frac 1n \ell''(\theta_0)} 
= \frac{\frac 1n \sum \ell_{X_i}'(\theta_0)}{- \frac 1n \sum \ell_{X_i}''(\theta_0)}
$$

$$
\sqrt{n}(\hat \theta - \theta_0) 
\approx 
\sqrt{n} \frac{\frac 1n \sum \ell_{X_i}'(\theta_0)}{- \frac 1n \sum \ell_{X_i}''(\theta_0)}
= \frac{\sqrt{n} (\bar Y - 0)}{- \frac 1n \sum \ell_{X_i}''(\theta_0)}
\implies \frac{\mathrm{Normal}(0, \mathrm{Var}[\ell'_{X_1}(\theta_0)])}{- \mathbb E[\ell_{X_1}'' (\theta_0)]}
= \frac{\mathrm{Normal}(0, \mathrm{Var}[\ell'_{X_1}(\theta_0)])}{\mathrm{Var}[\ell'_{X_1} (\theta_0)]}
$$

$$
\sim \mathrm{Normal}(0, \frac{1}{\mathrm{Var}[\ell'_{X_1} (\theta_0)]})
$$
and $\frac{1}{\mathrm{Var}[\ell'_{X_1} (\theta_0)]}$ is the CRLB.

$\implies$ MLE is unbiased and achieves CR lower bound variance aysmptotically.

frequentist mse

what is bayesian evaluator? frequentist mse assumes that theta is fixed, not random
bayesian analogue of mse --> E[(t(x) - Theta)^2], where Theta is a random var whose dist is a prior

can compute EV in 2 ways. 
1. E[ E[(f(x) - Theta)^2] | Theta] = int = frequentist mse weighted averaged over theta

Best estimator according to Bayesian MSE is the Bayesian estimator

2. EE t - Theta | X, choose t(x) to minimize inner EV = integral

t(x) = E[Theta | X]



root finding algorithm -- gamma dist, when you work out gamma MLE, can't actually find that value, for when there's no MLE closed form solution


stats 118 -- jensens, delta method, see application 


## 04/27


setting gradient of loss to 0, to find the variance of MLE, we have
$$
\sqrt n (
\begin{bmatrix}
\hat \alpha \\ \hat \lambda
\end{bmatrix} 
- 
\begin{bmatrix}
\alpha_0 \\ \lambda_0
\end{bmatrix}
)
\approx
\mathrm{MVN}(
\begin{bmatrix}
0 \\ 0
\end{bmatrix},
[\mathrm{Var}[\ell'(\alpha, \lambda)]]^{-1}

)
$$

where variance is a covariance matrix, or can do hessian for $\ell''$


Fisher information: 
$$
\mathrm{Var}[\ell'] = -\mathbb E[\ell'']
$$

Measure of curvature for $\ell$. more curvature means better information. 

### Uncertainty Measurements!

Suppose $X_1, \ldots, X_n$ are iid, Normal ($\mu, \sigma^2$)

where $\sigma^2$ is known. We want to estimate $\mu$ with pdf

$$
\frac{1}{\sigma \sqrt{2\pi}} \exp{-\frac{1}{2\sigma^2} (x - \mu)^2}
$$

a) what is the MLE of $\mu$? --> derivative of log likelihood, set to 0

b) Is the MLE unbiased? --> expected value

c) If so, it the best unbiased estimator? --> cramer rao lower bound


What if instead of reporting point estimate $\hat mu = \bar x$, we want to report an interval around $\bar x$?


<u>Confidence Intervals</u>

What is the distribution of $\hat \mu = \bar X$? 
$$
\bar X \sim \mathrm{Normal}(\mu_0, \frac{\sigma^2}{n})
$$

With standard z-score confidence intervals, we have 95% confidence for $\mu_0$ between
$$
\pm 2\sqrt{\frac{\sigma^2}{n}}
$$

If $\bar X$ truly is from $\mu$ dist, then
$$
\bar X \in \mu_0 \pm 2\sqrt{\frac{\sigma^2}{n}} \implies \mu_0 \in \bar X \pm 2\sqrt{\frac{\sigma^2}{n}} 
$$

Can say 95% chance that $\bar X \pm 2\sigma$ contains $\mu_0$, which is a statement about the procedure in which $\bar X$ is unknown and a random variable. However, we cannot say that there is a 95% chance that $(a, b)$ contains $\mu_0$, where a and b are measured numbers. This is because $(a,b)$ is now a fixed interval and is not random. The initial statement does not apply to a particular interval. So, we say


"We are 95% confident that $\mu_0$ is between 91.8 and 92.2."


Confidence is not the same as chance.

This is the fault of the frequentist interpretation. 